# WordPiece TOkenization for NER Task using the BioBERT's TOkezenizer


In [19]:
import json
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ========================
# LOAD TOKENIZER AND MODEL
# ========================
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

# ========================
# LOAD THE WORD TOKENIZED DATASET
# ========================
dataset = []
with open('data/synthetic_data_tokenized.jsonl', 'r') as f:
    for line in f:
        row = json.loads(line)
        dataset.append(json.loads(line))



Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Quickly check how the TOkenizer Works

In [20]:
# Load all data from JSONL into a list of dictionaries
ex1 = dataset[0]
print(ex1.keys())
print("Tokenize text")
toks = tokenizer(ex1['text'])
print(toks)

print("\nTokenize tokens")

print("Demonstrating is_split_into_words=False vs is_split_into_words=True:")
print("- Setting is_split_into_words=False assumes your input is a single string or a flat list of tokens as a string, so passing a list of word tokens (especially if nested) may not work as intended.")
print("- Setting is_split_into_words=True tells the tokenizer that the input is a list of words (not a string), so it will tokenize and align each word separately. This is typically required when you want to map predictions back to words in sequence labeling tasks. (If unsure: when working with a list of word tokens, True is likely what you want.)")

print("\nwith is_split_into_words=False (this will not work due to the nested lists)")
toks_ids = tokenizer(ex1['word_tokens'], is_split_into_words=False)
print(toks_ids)
# Will not work because of nested lists
# print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

print("\nwith is_split_into_words=True")
toks_ids = tokenizer(ex1['word_tokens'], is_split_into_words=True)
print(toks_ids)
print("Word ids:\n", toks_ids.word_ids())
print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

# [CLS] -> Classification token (start of sequence)
# [SEP] -> Separator token (end of sequence)

# Tokenization flow: words -> word ids -> token ids

dict_keys(['text', 'word_tokens', 'word_labels', 'symptom_id', 'is_negated'])
Tokenize text
{'input_ids': [101, 1175, 1132, 1185, 8006, 1104, 188, 19091, 8380, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Tokenize tokens
Demonstrating is_split_into_words=False vs is_split_into_words=True:
- Setting is_split_into_words=False assumes your input is a single string or a flat list of tokens as a string, so passing a list of word tokens (especially if nested) may not work as intended.
- Setting is_split_into_words=True tells the tokenizer that the input is a list of words (not a string), so it will tokenize and align each word separately. This is typically required when you want to map predictions back to words in sequence labeling tasks. (If unsure: when working with a list of word tokens, True is likely what you want.)

with is_split_into_words=False (this will not work due to the nested lists)
{'input_ids': [[101, 11

# Tokenization

-> Assign -100 to special tokens:  [CLS], [SEP], pytorch will ignore predictions of these tokens


In [21]:
# Run an example of how the tokenizer workds
ex = dataset[10]
print("Sample text: ", ex)
word_tokens = ex['word_tokens']
print("Word tokens: ", word_tokens)
print("\n-------- TOKENIZE ! --------")
# TOKENIZE!
toks_ids = tokenizer(ex['word_tokens'],is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(toks_ids['input_ids'])
print("\nTokenized words (is_split_into_words=True):\n\t", tokens)
print("Each token id:\n\t", toks_ids['input_ids'])
word_ids = toks_ids.word_ids()
print("Mappings of tokens to word index in the list of word_tokens:\n\t", word_ids)


Sample text:  {'text': 'Reports periumbilic pelvic lump.', 'word_tokens': ['Reports', 'periumbilic', 'pelvic', 'lump', '.'], 'word_labels': ['O', 'B-SYMPTOM_s0574_POS', 'I-SYMPTOM_s0574_POS', 'I-SYMPTOM_s0574_POS', 'O'], 'symptom_id': 's0574', 'is_negated': False}
Word tokens:  ['Reports', 'periumbilic', 'pelvic', 'lump', '.']

-------- TOKENIZE ! --------

Tokenized words (is_split_into_words=True):
	 ['[CLS]', 'reports', 'per', '##ium', '##bil', '##ic', 'p', '##el', '##vic', 'lump', '.', '[SEP]']
Each token id:
	 [101, 3756, 1679, 3656, 15197, 1596, 185, 1883, 15901, 16401, 119, 102]
Mappings of tokens to word index in the list of word_tokens:
	 [None, 0, 1, 1, 1, 1, 2, 2, 2, 3, 4, None]


In [22]:
# ===========================================================================
# Create wordpiece-tokenized dataset 
# ===========================================================================

# Prepare list for storing tokenized samples (optional, for downstream use)
tokenized_samples = []

for row in dataset:

    # Tokenize word tokens for each sample
    words = row['word_tokens']
    toks_ids = tokenizer(words, is_split_into_words=True)
    input_ids = toks_ids['input_ids']
    word_ids = toks_ids.word_ids()

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    token_labels = []
    previous_word_id = None

    for i, tok in enumerate(input_ids):
        word_id = word_ids[i]

        # Special tokens
        if word_id is None:
            token_labels.append("None") # -100
        # First subword of a word
        elif word_id != previous_word_id:
            token_labels.append(row['word_labels'][word_id])
        # Continuation subwords
        else:
            original_label = row['word_labels'][word_id]
            # If it's a B- tag, convert it to I-
            if isinstance(original_label, str) and original_label.startswith("B-"):
                fixed_label = original_label.replace("B-", "I-")
            else:
                fixed_label = original_label
            token_labels.append(fixed_label)
        previous_word_id = word_id

    # Prepare record for saving
    tokenized_sample = {
        "text": row["text"],
        "word_tokens": words,
        "word_labels": row["word_labels"],
        "tokens": tokens,
        "input_ids": input_ids,
        "token_labels": token_labels
    }
    tokenized_samples.append(tokenized_sample)

print("Example tokenized sample:")
print(tokenized_samples[0])


Example tokenized sample:
{'text': 'There are no symptoms of stridor.', 'word_tokens': ['There', 'are', 'no', 'symptoms', 'of', 'stridor', '.'], 'word_labels': ['O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'O'], 'tokens': ['[CLS]', 'there', 'are', 'no', 'symptoms', 'of', 's', '##tri', '##dor', '.', '[SEP]'], 'input_ids': [101, 1175, 1132, 1185, 8006, 1104, 188, 19091, 8380, 119, 102], 'token_labels': ['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'O', 'None']}


In [23]:
with open("data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in tokenized_samples:
        f.write(json.dumps(sample) + "\n")

# Create ids2label and labels2ids mappings

In [24]:
dataset = []
with open("data/data_wordpiece_tokenized_biobert.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))
        

In [25]:
unique_labels = set()

for row in dataset: 
    for lbl in row["token_labels"]:
        if lbl != "None":  # ignore special tokens
            unique_labels.add(lbl)

# Sort for stable ordering
unique_labels = sorted(list(unique_labels))
# Create mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Number of labels:", len(label2id))

# Save mapping:
with open("data/label2id_biobert.json", "w") as f:
    json.dump(label2id, f, indent = 2)
with open("data/id2label_biobert.json", "w") as f:
    json.dump(id2label, f, indent = 2)

Number of labels: 3525


In [27]:
# Convert labels to numeric IDs
for row in dataset:
    row["token_label_ids"] = [
        -100 if lbl == "None" else label2id[lbl]
        for lbl in row["token_labels"]
    ]

with open("data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in dataset:
        f.write(json.dumps(sample) + "\n")

In [28]:
# Sanity check!
print(dataset[0]["token_labels"])
print(dataset[0]["token_label_ids"])

['None', 'O', 'O', 'O', 'O', 'O', 'B-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'I-SYMPTOM_s0486_NEG', 'O', 'None']
[-100, 3524, 3524, 3524, 3524, 3524, 970, 2742, 2742, 3524, -100]


In [29]:
len(dataset[0]["token_labels"]), len(dataset[0]["token_label_ids"])

(11, 11)

In [30]:
# Compare the differences between the id2label 

with open("data/id2label.json", "r") as f:
    id2label_distill = json.load(f)
with open("data/id2label_biobert.json", "r") as f:
    id2label_biobert = json.load(f)

In [32]:
# Compare keys (IDs) that are in one dict but not the other
ids_in_distill_not_biobert = set(id2label_distill.values()) - set(id2label_biobert.values())
ids_in_biobert_not_distill = set(id2label_biobert.values()) - set(id2label_distill.values())

print("Labels in id2label_distill but not in id2label_biobert:", ids_in_distill_not_biobert)
print("Labels in id2label_biobert but not in id2label_distill:", ids_in_biobert_not_distill)

Labels in id2label_distill but not in id2label_biobert: set()
Labels in id2label_biobert but not in id2label_distill: {'I-SYMPTOM_s0115_POS', 'I-SYMPTOM_s0385_POS', 'I-SYMPTOM_s0311_NEG', 'I-SYMPTOM_s0841_POS', 'I-SYMPTOM_s0120_NEG', 'I-SYMPTOM_s0010_NEG', 'I-SYMPTOM_s0120_POS', 'I-SYMPTOM_s0841_NEG', 'I-SYMPTOM_s0035_NEG', 'I-SYMPTOM_s0268_POS', 'I-SYMPTOM_s0289_POS', 'I-SYMPTOM_s0010_POS', 'I-SYMPTOM_s0396_NEG', 'I-SYMPTOM_s0747_POS', 'I-SYMPTOM_s0396_POS', 'I-SYMPTOM_s0311_POS', 'I-SYMPTOM_s0667_POS', 'I-SYMPTOM_s0667_NEG', 'I-SYMPTOM_s0115_NEG', 'I-SYMPTOM_s0385_NEG', 'I-SYMPTOM_s0252_POS', 'I-SYMPTOM_s0268_NEG', 'I-SYMPTOM_s0252_NEG', 'I-SYMPTOM_s0035_POS', 'I-SYMPTOM_s0289_NEG', 'I-SYMPTOM_s0747_NEG'}
